<a href="https://colab.research.google.com/github/SZ2012-8CYL1ND3R/symmetrical-system/blob/main/Agentic_AI_Day_16.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [10]:
import requests

def get_detailed_weather(city, unit="Celsius"):
  # A dictionary mapping city names to their latitude and longitude
  city_coords = {
      "toronto": {"lat": 43.65, "lon": -79.38},
      "vancouver": {"lat": 49.28, "lon": -123.12},
      "london": {"lat": 51.50, "lon": -0.12},
      "tokyo": {"lat": 35.67, "lon": 139.65},
      "new york": {"lat": 40.71, "lon": -74.00}
  }

  city_key = city.lower().strip()
# Guard check if the city isn't in our dictionary.
  if city_key not in city_coords:
    return f"Sorry Salman, I don't have coordinate data for {city} in my database yet."

  lat = city_coords[city_key]["lat"]
  lon = city_coords[city_key]["lon"]

  # Dynamic f-string URL construction
  url = f"https://api.open-meteo.com/v1/forecast?latitude={lat}&longitude={lon}&current=temperature_2m"

  try:
    response = requests.get(url, timeout=5)

    if response.status_code != 200:
      return f"Error: Could not retrieve weather data (Server returned status code {response.status_code})."

    data = response.json()
    temp = data["current"]["temperature_2m"]

    if unit.lower() == "fahrenheit":
      temp = (temp * 9/5) + 32

    return f"The live temperature in {city.capitalize()} is {temp}° {unit}."

  except requests.exceptions.RequestException as e:
    return f"Network Error: Unbale to connect to weather service. ({e})"

def control_car_climate(car, temperature):
  return f"Setting the climate control in your {car} to {temperature}°C."

tool_dispatcher = {
    "detailed_weather": get_detailed_weather,
    "car_climate": control_car_climate
}

conversation_history = [
    {"role": "system", "content": "You are Nova, an advanced, autonomous agent with live API capabilities."}
]

print("🤖 Nova Agent Initialized with Live API Integration.")
print("Type 'exit' or 'quit' at any time to end the session.\n")

while True:
  user_input = input("USER: ")

  if user_input.lower() in ["exit", "quit"]:
    print("ASSISTANT: Goodbye, Salman. Have a wonderful day.")
    break

  conversation_history.append({"role": "user", "content": user_input})

  if len(conversation_history) > 7:
    system_prompt = conversation_history[0]
    recent_messages = conversation_history[-6:]
    conversation_history = [system_prompt] + recent_messages
    print("⚙️ [System: Memory trimmed to keep agent fast.]")

  lower_input = user_input.lower()
  words = user_input.split()

  triggered_tools = []
  tool_results = []

  if "weather" in lower_input:
    city = "Toronto"
    unit = "Celsius"

    # Check for unit preference
    if "fahrenheit" in lower_input:
      unit = "Fahrenheit"

    for preposition in ["in", "at"]:
      if preposition in [w.lower() for w in words]:
        lower_words = [w.lower() for w in words]
        idx = lower_words.index(preposition)
        if idx + 1 < len(words):
          city = words[idx + 1].strip("?.,!")

    # Package into a structured dictionary with arguments
    triggered_tools.append({
        "tool": "detailed_weather",
        "args": {"city": city, "unit": unit}
    })

  if "climate" in lower_input or "ac" in lower_input:
    triggered_tools.append({
        "tool": "car_climate",
        "args": {"car": "Mercedes", "temperature": 21}
    })

  if not triggered_tools:
    fallback_message = "I'm sorry, I couldn't figure out which tool to use. Try asking about the live weather or car climate."
    conversation_history.append({"role": "assistant", "content": fallback_message})
    print(f"ASSISTANT: {fallback_message}\n")
    continue

  for item in triggered_tools:
    t_name = item["tool"]
    t_args = item["args"] # The dictionary package

    if t_name in tool_dispatcher:
      func = tool_dispatcher[t_name]
      # Python's ** unpacks the dictionary keys directly into function parameters
      result = func(**t_args)
      tool_results.append(result)
    else:
      tool_results.append(f"Error: tool {t_name} not found.")

  final_output = "  |  ".join(tool_results)
  conversation_history.append({"role": "assistant", "content": final_output})
  print(f"ASSISTANT: {final_output}\n")

🤖 Nova Agent Initialized with Live API Integration.
Type 'exit' or 'quit' at any time to end the session.

USER: What is the weather in Paris?
ASSISTANT: Sorry Salman, I don't have coordinate data for Paris in my database yet.

USER: What is the weather in New York?
ASSISTANT: Sorry Salman, I don't have coordinate data for New in my database yet.

USER: What is the weather in NewYork?
ASSISTANT: Sorry Salman, I don't have coordinate data for NewYork in my database yet.

USER: What is the weather in "New York"?
⚙️ [System: Memory trimmed to keep agent fast.]
ASSISTANT: Sorry Salman, I don't have coordinate data for "New in my database yet.

USER: exit
ASSISTANT: Goodbye, Salman. Have a wonderful day.
